In [ ]:
import os
import random
import hashlib
from datetime import datetime, timezone
import time
import pickle
import gc
import numpy as np
import pandas as pd
import importlib

import matplotlib.pyplot as plt


import tensorflow as tf
from tensorflow.keras import (
    layers, 
    models, 
    callbacks
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, 
    LabelEncoder, 
    MinMaxScaler
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_squared_error, 
    mean_absolute_error, 
    silhouette_score
)

import model_utils
from model_utils import *

importlib.reload(model_utils)

In [ ]:
# 1. Suppress TF internal C++ logs and disable oneDNN thread conflicts
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# 2. Limit CPU threading overhead on Windows to prevent kernel segmentation faults
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

print("✓ Environment variables set successfully.")



In [ ]:
# Configurations
# Experiment settings
NUM_CLIENTS_list = [10, 15, 20]  # Number of clients to simulate
IID_list = [True, False]  # Whether data is identically and independently distributed
SEED_list = [42, 123, 456]  # Random seeds for reproducibility

window_size = 10
local_batch_size = 32
ae_batch_size = 64
# ==========================================
# STAGE 1: Autoencoder (Feature Extraction)
# ==========================================
# The Autoencoder trains quickly in unsupervised mode to compress features.
ae_rounds = 20         # Server aggregates AE model 10 times total
local_ae_epochs = 3    # Each client runs 3 epochs per round on local data

# ==========================================
# STAGE 2: BiGRU (Intrusion Classifier)
# ==========================================
# The classifier requires more rounds to converge over Non-IID distributions.
bigru_rounds = 50      # Server aggregates BiGRU model 50 times total (matches Table 3)
local_bigru_epochs = 2 # Each client runs E epochs per round on local data


In [ ]:

def build_autoencoder(input_dim, bottleneck_dim=None):
    # Set bottleneck to ~50% if not specified to avoid over-compression
    if bottleneck_dim is None:
        bottleneck_dim = int(input_dim * 0.50)

    inp = layers.Input(shape=(input_dim,))

    # Encoder Branch
    x = layers.Dense(int(input_dim * 0.75), activation="leaky_relu")(inp)
    
    # Bottleneck Layer (Linear/LeakyReLU keeps negative activation values)
    encoded = layers.Dense(bottleneck_dim, activation="leaky_relu", name="encoded_features")(x)

    # Decoder Branch
    x = layers.Dense(int(input_dim * 0.75), activation="leaky_relu")(encoded)
    
    # Use 'sigmoid' if inputs are scaled [0, 1]; use 'linear' if standard-scaled (Z-score)
    decoded = layers.Dense(input_dim, activation="sigmoid")(x) 

    autoencoder = models.Model(inp, decoded, name="Autoencoder")
    encoder = models.Model(inp, encoded, name="Encoder")

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse"
    )

    return autoencoder, encoder

In [ ]:

def execute_phase_1_federated_ae(clients_data, input_dim, bottleneck_dim=None, rounds=20, local_epochs=3):
    """
    Executes Phase 1: Federated Training of Global Autoencoder
    """
    print("=== STARTING PHASE 1: FEDERATED AUTOENCODER TRAINING ===")    
    print('1. Instantiate Server Global Model & Clients')

    final_round_client_losses = []
    global_ae, global_encoder = build_autoencoder(input_dim, bottleneck_dim)
    global_weights = global_ae.get_weights()
    
    print('2. Loading data to clients')
    clients = []
    for i in range(len(clients_data)):
        print(f'Client {i+1}: Samples {len(clients_data[i])}')
        clients.append(ClientAE(i, clients_data[i], input_dim, bottleneck_dim, build_autoencoder))

    print()
    print('3. FL Rounds Execution')
    for r in range(1, rounds + 1):
        client_weights = []
        client_samples = []
        round_losses = []

        print(f'---ROUND {r}---')
        for idx, client in enumerate(clients):
            # Step 1: Push global weights to client
            client.set_weights(global_weights)
            
            # Step 2: Client trains locally
            w, n_samples, loss = client.train_local(local_epochs=local_epochs)
            
            client_weights.append(w)
            client_samples.append(n_samples)
            round_losses.append(loss)

            print(f"Client {idx+1}: Sample = {n_samples}, Loss = {loss}")

        if r == rounds:
            final_round_client_losses = round_losses

        # Step 3: Server aggregates local weights
        global_weights = federated_averaging(client_weights, client_samples)
        
        avg_loss = np.mean(round_losses)
        print(f"Phase 1 - Round {r}/{rounds} | Average Local Reconstruction MSE Loss: {avg_loss:.6f}")
        print()

    # Set final global weights
    global_ae.set_weights(global_weights)
    print("=== PHASE 1 COMPLETE: Global Autoencoder Converged ===\n")
    
    return global_ae, global_encoder, final_round_client_losses


In [ ]:
def build_bigru_attention_model(input_shape=(10, 37), learning_rate=1e-3):
    inputs = layers.Input(shape=input_shape, name="sequence_input")
    
    # Layer 1: First BiGRU Block
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.20), 
        name="bigru_1"
    )(inputs)
    
    # Layer 2: Second BiGRU Block (return_sequences=True for Attention)
    x = layers.Bidirectional(
        layers.GRU(64, return_sequences=True, dropout=0.20), 
        name="bigru_2"
    )(x)
    
    # Layer 3: Temporal Attention Pooling
    x = TemporalAttention(name="temporal_attention")(x)
    
    # Layer 4: Dense Classification Head
    x = layers.Dropout(0.20, name="dropout")(x)
    x = layers.Dense(32, activation="gelu", name="dense_32")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="attack_probability")(x)

    model = models.Model(inputs, outputs, name="federated_bigru_attention_ids")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

In [ ]:
# ============================================================
# Experiments
# ============================================================

for NUM_CLIENTS in NUM_CLIENTS_list:
    for IID in IID_list:
        for SEED in SEED_list:
            os.environ["PYTHONHASHSEED"] = str(SEED)
            random.seed(SEED)
            np.random.seed(SEED)
            tf.keras.utils.set_random_seed(SEED)

            # Optional: improves deterministic behavior, but may reduce speed
            try:
                tf.config.experimental.enable_op_determinism()
            except Exception:
                pass

            # 1. Load and Preprocess Tabular Data (2D)
            X_train_clients, y_train, X_test, y_test = prepare_data(r"ton-iot-train-test-network/train_test_network.csv", 
                num_clients=NUM_CLIENTS, 
                iid=IID, 
                alpha=0.5, 
                test_size=0.2
            )
            for c in range(NUM_CLIENTS):
                print(f"Client {c+1}\t{len(y_train[c])}\t{len(y_train[c][y_train[c]==0])}\t{len(y_train[c][y_train[c]==1])}")


            telemetry = {}

            print("X_train_clients[0] shape:", X_train_clients[0].shape)
            print("y_train[0] shape:", y_train[0].shape)
            print("X_test shape:", X_test.shape)
            print("y_test shape:", y_test.shape)

            input_dim = len(X_train_clients[0][0])  # 2D feature count (e.g., 42)

            # 2. Execute Phase 1: Federated Training of 2D Autoencoder
            # Pass the LIST of client arrays: X_train_clients
            global_ae, global_encoder, final_round_client_losses = execute_phase_1_federated_ae(
                clients_data=X_train_clients, 
                input_dim=input_dim, 
                bottleneck_dim=int(input_dim * 0.50), # e.g., compress 50%
                rounds=ae_rounds, 
                local_epochs=local_ae_epochs
            )

            # 3. Phase 1 Transition: Compress 2D features and create 3D Windows per Client
            X_clients, y_clients = [], []

            for i in range(NUM_CLIENTS):
                # Step A: Compress client 2D features using global encoder
                X_compressed = global_encoder.predict(X_train_clients[i])
                
                # Step B: Build 3D implicit temporal windows on compressed data
                X_seq_c, y_seq_c = build_implicit_temporal_windows(
                    X_compressed, 
                    y_train[i], 
                    sequence_length=window_size, 
                    step=1
                )
                
                X_clients.append(X_seq_c)
                y_clients.append(y_seq_c)

            # Verification
            print(f"Client 0 3D Tensor Shape for BiGRU: {X_clients[0].shape}")
            # Output will be: (N_sequences, 10, bottleneck_dim) -> Perfect for Phase 2!

            telemetry["data_distribution"] = {
                "raw_clients": {},
                "sequence_clients": {},
                "test_set": {},
                }


            # Encode global test set
            X_test_encoded = global_encoder.predict(
                X_test, batch_size=ae_batch_size, verbose=0
            )

            X_test_seq, y_test_seq = build_implicit_temporal_windows(
                X_test_encoded, y_test, sequence_length=window_size
            )

            # --------------------------------------------------------
            # Save Global Test Set Sequence Counts
            # --------------------------------------------------------
            test_norm_cnt = int((y_test_seq == 0).sum())
            test_att_cnt = int((y_test_seq == 1).sum())

            telemetry["data_distribution"]["test_set"] = {
                "normal": test_norm_cnt,
                "attack": test_att_cnt,
                "total": len(y_test_seq),
                "attack_ratio": float(test_att_cnt / len(y_test_seq)) if len(y_test_seq) > 0 else 0.0,
            }


            # --------------------------------------------------------
            # Step 4: Setup BiGRU Models & Logging
            # --------------------------------------------------------
            input_shape = X_test_seq.shape[1:]  # (window_size, latent_dim)

            global_bigru = build_bigru_attention_model(input_shape)
            worker_bigru = build_bigru_attention_model(input_shape)
            blockchain = ModelUpdateBlockchain()

            early_stop = tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=3, restore_best_weights=True
            )

            # 1. Parameter size tracking
            param_count = global_bigru.count_params()
            param_bytes_single = param_count * 4  # float32 = 4 bytes
            data_transferred_per_round_mb = (2 * NUM_CLIENTS * param_bytes_single) / (1024 * 1024)

            telemetry["num_params"] = param_count
            telemetry["param_bytes"] = param_bytes_single
            telemetry["rounds"] = []

            # 2. Add to telemetry dictionary structure
            telemetry["costs"] = {
                "params_count": param_count,
                "round_comm_mb": data_transferred_per_round_mb,
                "total_comm_mb": 0.0,
                "total_training_time_sec": 0.0,
            }

            # --------------------------------------------------------
            # Step 5: Phase 3 - Federated BiGRU Classifier Training
            # --------------------------------------------------------
            print(f"\n[5/6] Starting Phase 3: Federated BiGRU Training...")

            for round_idx in range(1, bigru_rounds + 1):
                print(
                    f"\n========== Federated Round {round_idx}/{bigru_rounds} =========="
                )

                round_start_time = time.time()

                client_round_logs = []

                global_weights = global_bigru.get_weights()
                local_weights = []
                local_sizes = []

                for client_id, (X_client, y_client) in enumerate(
                    zip(X_clients, y_clients), start=1
                ):
                    worker_bigru.set_weights(global_weights)

                    # 1. Sequential validation split (prevents temporal leakage)
                    val_split = int(len(X_client) * 0.8)
                    X_loc_train, X_loc_val = X_client[:val_split], X_client[val_split:]
                    y_loc_train, y_loc_val = y_client[:val_split], y_client[val_split:]

                    # 2. Local Training
                    client_history = worker_bigru.fit(
                        X_loc_train,
                        y_loc_train,
                        validation_data=(X_loc_val, y_loc_val),
                        epochs=local_bigru_epochs,
                        batch_size=local_batch_size,
                        verbose=0,
                        callbacks=[early_stop],
                        shuffle=False,  # Maintain sequential integrity
                    )

                    # 3. Predict local validation probabilities for client-level metrics
                    y_val_probs = worker_bigru.predict(X_loc_val, verbose=0).ravel()
                    c_metrics = compute_binary_metrics(y_loc_val, y_val_probs)
                    balanced_acc, f1_macro = compute_robust_metrics(y_loc_val, y_val_probs)
                    c_metrics["balanced_accuracy"] = balanced_acc
                    c_metrics["f1_macro"] = f1_macro
                    c_metrics["client_id"] = client_id
                    c_metrics["loss"] = float(client_history.history["val_loss"][-1])

                    client_round_logs.append(c_metrics)

                    # Save local weights for FedAvg
                    local_weights.append(worker_bigru.get_weights())
                    local_sizes.append(len(X_loc_train))

                    print(f"Client {client_id}: training completed (Acc: {balanced_acc:.4f}, F1: {f1_macro:0.4f})")

                # 4. Perform FedAvg Aggregation
                aggregated_weights = federated_averaging(local_weights, local_sizes)

                # Blockchain Validation
                new_block = blockchain.add_model_update(round_idx, aggregated_weights)
                if not blockchain.validate_chain():
                    raise RuntimeError(
                        f"Blockchain validation failed in round {round_idx}."
                    )

                global_bigru.set_weights(aggregated_weights)

                # 5. Global Model Evaluation on Held-Out Test Set
                global_test_probs = global_bigru.predict(X_test_seq, verbose=0).ravel()
                g_metrics = compute_binary_metrics(y_test_seq, global_test_probs)

                # Evaluate test loss
                global_loss = global_bigru.evaluate(X_test_seq, y_test_seq, verbose=0)[0]
                g_metrics["loss"] = float(global_loss)

                # 6. Append complete telemetry record for this round
                round_duration = time.time() - round_start_time

                # Store round convergence & cost data
                telemetry["rounds"].append({
                    "round": round_idx,
                    "global_metrics": g_metrics,
                    "client_metrics": client_round_logs,
                    "global_loss": global_loss,
                    "round_duration_sec": round_duration,
                    "cum_comm_mb": round_idx * data_transferred_per_round_mb,
                })

                telemetry["costs"]["total_comm_mb"] = round_idx * data_transferred_per_round_mb
                telemetry["costs"]["total_training_time_sec"] += round_duration

                print(
                    f"Round {round_idx} | loss={global_loss:.4f} | acc={g_metrics["accuracy"]:.4f} | "
                    f"prec={g_metrics["precision"]:.4f} | rec={g_metrics["recall"]:.4f} | f1={g_metrics["f1_score"]:.4f}"
                )

                del local_weights
                gc.collect()

            # --------------------------------------------------------
            # Step 6: Final Evaluation
            # --------------------------------------------------------
            print("\n[6/6] Final global model evaluation...")
            final_results = evaluate_binary_model(global_bigru, X_test_seq, y_test_seq)
            telemetry["final_results"] = final_results

            # 4. Re-run Phase 1 Evaluation
            metrics = evaluate_phase1_autoencoder(
                global_ae=global_ae,
                global_encoder=global_encoder,
                X_test=X_test,
                y_test=y_test,
                feature_names=None,
                client_losses_round_final=final_round_client_losses
            )

            plot_federated_convergence(final_round_client_losses)

            iid_str='IID' if IID else "Non-IID"
            file_name = f"experiments/TON-n_{NUM_CLIENTS}-{iid_str}-seed_{SEED}.pkl"
            results = {
                'global_ae': global_ae,
                'global_encoder': global_encoder,
                'global_bigru': global_bigru,
                'telemetry': telemetry,
                'metrics': metrics
            }

            with open(file_name, 'wb') as f:
                pickle.dump(results, f)
